# 📡 Streaming — LangChain

This notebook explores **streaming** in LangChain: getting a model's response token-by-token as it's generated, instead of waiting for the full output — and how to stream from a full tool-calling **agent**, not just a plain LLM call.

---

## 🧭 What You'll Build

1. **Streaming Basics** — `.astream()`, `AIMessageChunk`, and reassembling a full message
2. **A Tool-Calling Agent, Ready to Stream** — tools, prompt, and a synchronous `CustomAgentExecutor`
3. **Enabling Token-Level Streaming** — configurable callbacks + a custom `QueueCallbackHandler`
4. **Testing the Streaming Executor** — an async, streaming `CustomAgentExecutor`, run and parsed several ways


## 1. Streaming Basics — `.astream()`

### Setting Up the LLM

Configure the Groq-hosted `llama-3.3-70b-versatile` model.


In [ ]:
import os

from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "put your api key"


model="llama-3.3-70b-versatile"



# For normal accurate responses
llm = ChatGroq(temperature=0.0, model=model)

**Baseline:** call `.invoke()` normally — the full response arrives in one go, only after the model has finished generating.


In [2]:
llm_out = llm.invoke("Hello there")
llm_out

AIMessage(content="Hello. It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 37, 'total_tokens': 62, 'completion_time': 0.052431933, 'completion_tokens_details': None, 'prompt_time': 0.00089677, 'prompt_tokens_details': None, 'queue_time': 0.051950751, 'total_time': 0.053328703}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f72c4-eaa4-7ad0-91af-512a7b05a992-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 25, 'total_tokens': 62})

**Streaming instead:** `llm.astream(...)` is an async generator that yields the response **token-by-token**, as it's generated, instead of waiting for the full message.


In [3]:
tokens = []
async for token in llm.astream("What is NLP?"):
    tokens.append(token)
    print(token.content, end="|", flush=True)

|**|Natural| Language| Processing| (|N|LP|)**| is| a| sub|field| of| artificial| intelligence| (|AI|)| that| deals| with| the| interaction| between| computers| and| humans| in| natural| language|.| It| is| a| multid|isc|iplinary| field| that| combines| computer| science|,| lingu|istics|,| and| cognitive| psychology| to| enable| computers| to| process|,| understand|,| and| generate| human| language|.

|**|Key| As|pects| of| N|LP|:|**

|1|.| **|Text| Processing|**:| N|LP| involves| processing| and| analyzing| large| amounts| of| text| data|,| including| token|ization|,| stemming|,| and| le|mm|at|ization|.
|2|.| **|Language| Understanding|**:| N|LP| aims| to| enable| computers| to| understand| the| meaning| and| context| of| human| language|,| including| syntax|,| semantics|,| and| prag|m|atics|.
|3|.| **|Language| Generation|**:| N|LP| involves| generating| human|-like| language|,| including| text|,| speech|,| and| dialogue|.
|4|.| **|Machine| Learning|**:| N|LP| often| employs| machine|

Each yielded item is an `AIMessageChunk` — a partial piece of the final `AIMessage`.


In [4]:

tokens[0]

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'groq'}, id='lc_run--019f72c4-f026-7b50-81b7-eb53c403a732', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

In [5]:

tokens[1]

AIMessageChunk(content='**', additional_kwargs={}, response_metadata={'model_provider': 'groq'}, id='lc_run--019f72c4-f026-7b50-81b7-eb53c403a732', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

**Reassembling the message:** `AIMessageChunk` objects support `+`, so adding consecutive chunks together in order reconstructs the correct partial (or full) message.


In [6]:
tokens[0] + tokens[1] + tokens[2] + tokens[3] + tokens[4]

AIMessageChunk(content='**Natural Language Processing', additional_kwargs={}, response_metadata={'model_provider': 'groq'}, id='lc_run--019f72c4-f026-7b50-81b7-eb53c403a732', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

Order matters — adding the same chunks in **reverse** produces a different (and generally incorrect) result, since each chunk is only meaningful relative to what came immediately before it.


In [7]:
tokens[4] + tokens[3] + tokens[2] + tokens[1] + tokens[0]

AIMessageChunk(content=' Processing LanguageNatural**', additional_kwargs={}, response_metadata={'model_provider': 'groq'}, id='lc_run--019f72c4-f026-7b50-81b7-eb53c403a732', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

## 2. A Tool-Calling Agent, Ready to Stream

Define the calculator tools plus a dedicated `final_answer` tool — the same pattern used in the agents notebook — so there's an agent worth streaming from.


In [8]:
from langchain_core.tools import tool

@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' and 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the power of 'y'."""
    return x ** y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract 'x' from 'y'."""
    return y - x

@tool
def final_answer(answer: str, tools_used: list[str]) -> str:
    """Use this tool to provide a final answer to the user.
    The answer should be in natural language as this will be provided
    to the user directly. The tools_used must include a list of tool
    names that were used within the `scratchpad`. You MUST use this tool
    to conclude the interaction.
    """
    return {"answer": answer, "tools_used": tools_used}

Collect all tools, including `final_answer`, into a single list.


In [9]:
tools = [add, multiply, exponentiate, subtract, final_answer]

### The Agent Prompt

System rules + `chat_history` + `{input}` + `agent_scratchpad`, exactly as in the standard tool-calling agent.


In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You're a helpful assistant. When answering a user's question "
        "you should first use one of the tools provided. After using a "
        "tool the tool output will be provided back to you. You MUST "
        "then use the final_answer tool to provide a final answer to the user. "
        "DO NOT use the same tool more than once."
    )),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

### The Agent Runnable

Build the LCEL agent pipeline, forcing tool use via `tool_choice="any"`.


In [11]:
from langchain_core.runnables.base import RunnableSerializable

tools = [add, subtract, multiply, exponentiate, final_answer]

# define the agent runnable
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")
)

### `CustomAgentExecutor` (Synchronous)

The same hand-built agent loop from the agents notebook: call the agent, execute whichever tool it picks, append the result to the scratchpad, and stop once `final_answer` is called.


In [12]:
import json
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage


# create tool name to function mapping
name2tool = {tool.name: tool.func for tool in tools}

class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 3):
        self.chat_history = []
        self.max_iterations = max_iterations
        self.agent: RunnableSerializable = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
            }
            | prompt
            | llm.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
        )

    def invoke(self, input: str) -> dict:
        # invoke the agent but we do this iteratively in a loop until
        # reaching a final answer
        count = 0
        agent_scratchpad = []
        while count < self.max_iterations:
            # invoke a step for the agent to generate a tool call
            out = self.agent.invoke({
                "input": input,
                "chat_history": self.chat_history,
                "agent_scratchpad": agent_scratchpad
            })
            # if the tool call is the final answer tool, we stop
            if out.tool_calls[0]["name"] == "final_answer":
                break
            agent_scratchpad.append(out)  # add tool call to scratchpad
            # otherwise we execute the tool and add it's output to the agent scratchpad
            tool_out = name2tool[out.tool_calls[0]["name"]](**out.tool_calls[0]["args"])
            # add the tool output to the agent scratchpad
            action_str = f"The {out.tool_calls[0]['name']} tool returned {tool_out}"
            agent_scratchpad.append({
                "role": "tool",
                "content": action_str,
                "tool_call_id": out.tool_calls[0]["id"]
            })
            # add a print so we can see intermediate steps
            print(f"{count}: {action_str}")
            count += 1
        # add the final output to the chat history
        final_answer = out.tool_calls[0]["args"]
        # this is a dictionary, so we convert it to a string for compatibility with
        # the chat history
        final_answer_str = json.dumps(final_answer)
        self.chat_history.append({"input": input, "output": final_answer_str})
        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer_str)
        ])
        # return the final answer in dict form
        return final_answer

agent_executor = CustomAgentExecutor()

**Test it:** run a full query through the synchronous executor — no streaming yet, just the final result.


In [13]:

agent_executor.invoke(input="What is 10 + 10")

0: The add tool returned 20


{'answer': 'The answer to 10 + 10 is 20.', 'tools_used': ['add']}

## 3. Enabling Token-Level Streaming

`streaming=True` turns on token streaming for the LLM. `.configurable_fields(callbacks=...)` exposes `callbacks` as something that can be swapped in per-invocation — needed later to plug in a custom streaming handler.


In [14]:
from langchain_core.runnables import ConfigurableField

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    streaming=True
).configurable_fields(
    callbacks=ConfigurableField(
        id="callbacks",
        name="callbacks",
        description="A list of callbacks to use for streaming",
    )
)

A normal `.invoke()` still works exactly as before; the configurable field doesn't change default behavior.


In [15]:
llm.invoke("hi")

AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'model_provider': 'groq'}, id='lc_run--019f72c5-1097-79e0-8400-8c7b429dd687', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 23, 'total_tokens': 59})

Rebuild the agent runnable using this new streaming-enabled, configurable LLM.


In [16]:
# define the agent runnable
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")
)

### `QueueCallbackHandler`

A custom async callback handler that pushes every streamed token into an `asyncio.Queue` as it arrives, and detects when the `final_answer` tool has been called — so it knows when to signal that streaming is truly done (`<<DONE>>`) versus just the end of one intermediate step (`<<STEP_END>>`).


In [17]:
import asyncio
from langchain_core.callbacks.base import AsyncCallbackHandler


class QueueCallbackHandler(AsyncCallbackHandler):
    """Callback handler that puts tokens into a queue."""

    def __init__(self, queue: asyncio.Queue):
        self.queue = queue
        self.final_answer_seen = False

    async def __aiter__(self):
        while True:
            if self.queue.empty():
                await asyncio.sleep(0.1)
                continue
            token_or_done = await self.queue.get()

            if token_or_done == "<<DONE>>":
                # this means we're done
                return
            if token_or_done:
                yield token_or_done

    async def on_llm_new_token(self, *args, **kwargs) -> None:
        """Put new token in the queue."""
        #print(f"on_llm_new_token: {args}, {kwargs}")
        chunk = kwargs.get("chunk")
        if chunk:
            # check for final_answer tool call
            if tool_calls := chunk.message.additional_kwargs.get("tool_calls"):
                if tool_calls[0]["function"]["name"] == "final_answer":
                    # this will allow the stream to end on the next `on_llm_end` call
                    self.final_answer_seen = True
        await self.queue.put(chunk)
        return

    async def on_llm_end(self, *args, **kwargs) -> None:
        """Put None in the queue to signal completion."""
        #print(f"on_llm_end: {args}, {kwargs}")
        # this should only be used at the end of our agent execution, however LangChain
        # will call this at the end of every tool call, not just the final tool call
        # so we must only send the "done" signal if we have already seen the final_answer
        # tool call
        if self.final_answer_seen:
            await self.queue.put("<<DONE>>")
        else:
            await self.queue.put("<<STEP_END>>")
        return

**Test it:** stream a raw agent call — attach the handler via `.with_config({"callbacks": [streamer]})`, then iterate over `astream(...)` and print each raw chunk as it arrives.


In [18]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

tokens = []

async def stream(query: str):
    response = agent.with_config(
        {"callbacks": [streamer]}
    )
    async for token in response.astream({
        "input": query,
        "chat_history": [],
        "agent_scratchpad": []
    }):
        tokens.append(token)
        print(token, flush=True)

await stream("What is 10 + 10 and 10 - 10")

content='' additional_kwargs={} response_metadata={'model_provider': 'groq'} id='lc_run--019f72c5-1627-7083-abd6-b150c9fc4339' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={'tool_calls': [{'index': 0, 'id': 'b9ma4ck3e', 'function': {'arguments': '{"x":10,"y":10}', 'name': 'add'}, 'type': 'function'}]} response_metadata={'model_provider': 'groq'} id='lc_run--019f72c5-1627-7083-abd6-b150c9fc4339' tool_calls=[{'name': 'add', 'args': {'x': 10, 'y': 10}, 'id': 'b9ma4ck3e', 'type': 'tool_call'}] invalid_tool_calls=[] tool_call_chunks=[{'name': 'add', 'args': '{"x":10,"y":10}', 'id': 'b9ma4ck3e', 'index': 0, 'type': 'tool_call_chunk'}]
content='' additional_kwargs={'tool_calls': [{'index': 1, 'id': 'x7zngpmx0', 'function': {'arguments': '{"x":10,"y":10}', 'name': 'subtract'}, 'type': 'function'}]} response_metadata={'model_provider': 'groq'} id='lc_run--019f72c5-1627-7083-abd6-b150c9fc4339' tool_calls=[{'name': 'subtract', 'args': {'x': 10, 'y': 10}, 'i

**Reassemble the message:** add all the streamed chunks together to reconstruct the complete tool-call message.


In [19]:
tk = tokens[0]

for token in tokens[1:]:
    tk += token

tk

AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'b9ma4ck3e', 'function': {'arguments': '{"x":10,"y":10}', 'name': 'add'}, 'type': 'function'}, {'index': 1, 'id': 'x7zngpmx0', 'function': {'arguments': '{"x":10,"y":10}', 'name': 'subtract'}, 'type': 'function'}, {'index': 2, 'id': 'tf79x2s7z', 'function': {'arguments': '{"answer":"The result of 10 + 10 is 20 and 10 - 10 is 0.","tools_used":["add","subtract"]}', 'name': 'final_answer'}, 'type': 'function'}]}, response_metadata={'model_provider': 'groq', 'finish_reason': 'tool_calls', 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand'}, id='lc_run--019f72c5-1627-7083-abd6-b150c9fc4339', tool_calls=[{'name': 'add', 'args': {'x': 10, 'y': 10}, 'id': 'b9ma4ck3e', 'type': 'tool_call'}, {'name': 'subtract', 'args': {'x': 10, 'y': 10}, 'id': 'x7zngpmx0', 'type': 'tool_call'}, {'name': 'final_answer', 'args': {'answer': 'The result of 10 + 10 is 20 and 10 - 

### `CustomAgentExecutor` (Asynchronous, Streaming)

An async version of the executor: instead of a single blocking call per step, it streams each step token-by-token via `astream()`, accumulates the chunks into a full `AIMessage`, then executes whichever tool was called — repeating until `final_answer` is reached.


In [20]:
from langchain_core.messages import ToolMessage

class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 3):
        self.chat_history = []
        self.max_iterations = max_iterations
        self.agent: RunnableSerializable = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
            }
            | prompt
            | llm.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
        )

    async def invoke(self, input: str, streamer: QueueCallbackHandler, verbose: bool = False) -> dict:
        # invoke the agent but we do this iteratively in a loop until
        # reaching a final answer
        count = 0
        agent_scratchpad = []
        while count < self.max_iterations:
            # invoke a step for the agent to generate a tool call
            async def stream(query: str):
                response = self.agent.with_config(
                    callbacks=[streamer]
                )
                # we initialize the output dictionary that we will be populating with
                # our streamed output
                output = None
                # now we begin streaming
                async for token in response.astream({
                    "input": query,
                    "chat_history": self.chat_history,
                    "agent_scratchpad": agent_scratchpad
                }):
                    if output is None:
                        output = token
                    else:
                        # we can just add the tokens together as they are streamed and
                        # we'll have the full response object at the end
                        output += token
                    if token.content != "":
                        # we can capture various parts of the response object
                        if verbose: print(f"content: {token.content}", flush=True)
                    tool_calls = token.additional_kwargs.get("tool_calls")
                    if tool_calls:
                        if verbose: print(f"tool_calls: {tool_calls}", flush=True)
                        tool_name = tool_calls[0]["function"]["name"]
                        if tool_name:
                            if verbose: print(f"tool_name: {tool_name}", flush=True)
                        arg = tool_calls[0]["function"]["arguments"]
                        if arg != "":
                            if verbose: print(f"arg: {arg}", flush=True)
                return AIMessage(
                    content=output.content,
                    tool_calls=output.tool_calls,
                    tool_call_id=output.tool_calls[0]["id"]
                )

            tool_call = await stream(query=input)
            # add initial tool call to scratchpad
            agent_scratchpad.append(tool_call)
            # otherwise we execute the tool and add it's output to the agent scratchpad
            tool_name = tool_call.tool_calls[0]["name"]
            tool_args = tool_call.tool_calls[0]["args"]
            tool_call_id = tool_call.tool_call_id
            tool_out = name2tool[tool_name](**tool_args)
            # add the tool output to the agent scratchpad
            tool_exec = ToolMessage(
                content=f"{tool_out}",
                tool_call_id=tool_call_id
            )
            agent_scratchpad.append(tool_exec)
            count += 1
            # if the tool call is the final answer tool, we stop
            if tool_name == "final_answer":
                break
        # add the final output to the chat history, we only add the "answer" field
        final_answer = tool_out["answer"]
        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer)
        ])
        # return the final answer in dict form
        return tool_args

agent_executor = CustomAgentExecutor()

## 4. Testing the Streaming Executor

**Test it (verbose):** run a query and print every intermediate token, tool call, and argument as they stream in.


In [23]:

queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

out = await agent_executor.invoke("What is 10 + 10", streamer, verbose=True)

tool_calls: [{'index': 0, 'id': 'f3bp2eddk', 'function': {'arguments': '{"x":10,"y":10}', 'name': 'add'}, 'type': 'function'}]
tool_name: add
arg: {"x":10,"y":10}
tool_calls: [{'index': 0, 'id': '7147me1ab', 'function': {'arguments': '{"answer":"The answer to 10 + 10 is 20.","tools_used":["add"]}', 'name': 'final_answer'}, 'type': 'function'}]
tool_name: final_answer
arg: {"answer":"The answer to 10 + 10 is 20.","tools_used":["add"]}


**Test it (quiet):** the same call, without the verbose token-by-token printout — only the tool-call processing runs, silently.


In [25]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

out = await agent_executor.invoke("What is 10 + 10", streamer)

**Test it (concurrently):** run the executor as a background `asyncio.Task` while separately iterating over the `streamer` in real time — this is the pattern a real streaming API (e.g. FastAPI + SSE) would use to forward tokens to a client as they arrive.


In [26]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

task = asyncio.create_task(agent_executor.invoke("What is 10 + 10", streamer))

async for token in streamer:
    print(token, flush=True)

await task

message=AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'groq'}, id='lc_run--019f72dc-578b-7fe0-8b2f-aadf55c37fe9', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])
message=AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': '6aakyq9q7', 'function': {'arguments': '{"x":10,"y":10}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'model_provider': 'groq'}, id='lc_run--019f72dc-578b-7fe0-8b2f-aadf55c37fe9', tool_calls=[{'name': 'add', 'args': {'x': 10, 'y': 10}, 'id': '6aakyq9q7', 'type': 'tool_call'}], invalid_tool_calls=[], tool_call_chunks=[{'name': 'add', 'args': '{"x":10,"y":10}', 'id': '6aakyq9q7', 'index': 0, 'type': 'tool_call_chunk'}])
generation_info={'finish_reason': 'tool_calls', 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand'} message=AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls',

{'answer': 'The answer to 10 + 10 is 20', 'tools_used': ['add']}

**Parse the stream for display:** inspect each token for a `<<STEP_END>>` marker (print a newline) or for `tool_calls` (print the tool name once, then stream its arguments as they're generated) — turning the raw token stream into a clean, readable console log.


In [27]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

task = asyncio.create_task(agent_executor.invoke("What is 10 + 10", streamer))

async for token in streamer:
    # first identify if we have a <<STEP_END>> token
    if token == "<<STEP_END>>":
        print("\n", flush=True)
    # we'll first identify if the token is a tool call
    elif tool_calls := token.message.additional_kwargs.get("tool_calls"):
        # if we have a tool call with a tool name, we'll print it
        if tool_name := tool_calls[0]["function"]["name"]:
            print(f"Calling {tool_name}...", flush=True)
        # if we have a tool call with arguments, we ad them to our args string
        if tool_args := tool_calls[0]["function"]["arguments"]:
            print(f"{tool_args}", end="", flush=True)

_ = await task

Calling add...
{"x":10,"y":10}

Calling final_answer...
{"answer":"The answer to 10 + 10 is 20","tools_used":["add"]}